# Phase 0 — Colab 초기 세팅

이 노트북은 **한 번만** 실행하면 되는 환경 구축 절차입니다.
세션이 끊긴 뒤에는 `1. 런타임 확인` ~ `4. 패키지 설치`까지만 다시 실행하면 됩니다.

**실행 순서**
1. 런타임 확인 (GPU 할당 검증)
2. Google Drive 마운트
3. 프로젝트 코드 배치
4. 패키지 설치
5. ComMU 데이터셋 다운로드
6. W&B 연동
7. 데이터 sanity check
8. 세팅 결과 기록

---
⚠️ 시작 전: 메뉴에서 **런타임 > 런타임 유형 변경 > A100 GPU** 선택

## 1. 런타임 확인

GPU가 무엇으로 잡혔는지에 따라 `configs/base.yaml`의 `precision` 값이 달라집니다.
- **A100 / L4** → `bf16`
- **T4 / V100** → `fp16`

In [ ]:
!nvidia-smi

import sys, platform
print()
print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

import torch
print("torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
if torch.cuda.is_available():
    print("bf16 지원:", torch.cuda.is_bf16_supported())
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM   : {total:.1f} GB")

## 2. Google Drive 마운트

체크포인트와 데이터는 반드시 Drive에 저장합니다.
Colab 세션이 끊기면 `/content` 아래는 전부 사라지기 때문입니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/music-ai-portfolio')
for sub in ['data/raw', 'data/processed', 'checkpoints', 'outputs', 'docs/samples']:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

print("Drive 루트:", DRIVE_ROOT)
!ls -la "{DRIVE_ROOT}"

## 3. 프로젝트 코드 배치

**방법 A (권장)**: GitHub 레포를 만들고 clone
**방법 B**: 스캐폴드 zip을 Drive에 올려두고 압축 해제

아래는 방법 A 기준입니다. 처음이라면 먼저 GitHub에 빈 레포를 만들고,
로컬에서 스캐폴드를 push한 뒤 URL을 넣으세요.

In [ ]:
# ── 방법 A: GitHub clone ──────────────────────────────
GITHUB_URL = "https://github.com/leetin234/music-ai-portfolio.git"

%cd /content
if GITHUB_URL:
    !git clone {GITHUB_URL} music-ai-portfolio
    %cd /content/music-ai-portfolio
else:
    print("GITHUB_URL이 비어 있습니다. 방법 B 셀을 사용하세요.")

In [ ]:
# ── 방법 B: Drive에 올려둔 zip 압축 해제 ────────────────
# Drive의 music-ai-portfolio/ 아래에 scaffold zip을 올려둔 경우

# !unzip -q "/content/drive/MyDrive/music-ai-portfolio/music-ai-portfolio-scaffold.zip" -d /content/
# %cd /content/music-ai-portfolio
# !ls -la

## 4. 패키지 설치

⚠️ **torch는 절대 재설치하지 마세요.** Colab에 이미 CUDA 빌드가 깔려 있는데
pip로 다시 설치하면 CPU 버전으로 덮어써져서 학습이 안 됩니다.
`requirements.txt`에서 torch를 주석 처리해둔 이유입니다.

In [ ]:
# 시스템 패키지 (fluidsynth 등) — Phase 5 전까지는 선택 사항
# !bash scripts/setup_colab.sh

# 파이썬 패키지
!pip install -q -r requirements.txt

print("\n설치 확인 ─────────────────")
import importlib
for name in ["miditoolkit", "pretty_midi", "music21", "numpy", "pandas", "wandb", "yaml", "gradio"]:
    try:
        m = importlib.import_module(name)
        v = getattr(m, "__version__", "?")
        print(f"  ✅ {name:<14} {v}")
    except Exception as e:
        print(f"  ❌ {name:<14} {type(e).__name__}: {e}")

## 5. ComMU 데이터셋 다운로드

POZAlabs의 공식 레포에서 데이터만 가져옵니다.

> **왜 코드는 안 쓰나?**
> 공식 레포는 Python 3.8.12를 요구합니다. 최신 Colab 런타임과 맞지 않고,
> 우리는 전처리·모델을 직접 구현하는 것이 프로젝트의 핵심이므로
> **MIDI 파일과 메타데이터 CSV만** 사용합니다.

> **라이선스**: ComMU 데이터셋은 CC BY-NC-SA 4.0으로 배포됩니다.
> 비상업적 이용이며, 파생 결과물도 동일 조건으로 공유해야 합니다.
> README에 반드시 출처와 라이선스를 명시하세요.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/POZAlabs/ComMU-code.git

import os
os.makedirs('/content/ComMU-code/dataset/extracted', exist_ok=True)
!tar -xf /content/ComMU-code/dataset/commu_midi.tar -C /content/ComMU-code/dataset/

!echo "── 디렉토리 구조 ──" && ls /content/ComMU-code/dataset/
!echo && echo "── train raw 파일 수 ──"
!ls /content/ComMU-code/dataset/commu_midi/train/raw | wc -l
!echo "── val raw 파일 수 ──"
!ls /content/ComMU-code/dataset/commu_midi/val/raw | wc -l

In [ ]:
# Drive로 복사 (다음 세션부터 재다운로드 불필요)
import shutil
from pathlib import Path

SRC = Path('/content/ComMU-code/dataset')
DST = Path('/content/drive/MyDrive/music-ai-portfolio/data/raw')

if not (DST / 'commu_meta.csv').exists():
    shutil.copy(SRC / 'commu_meta.csv', DST / 'commu_meta.csv')
    print("✅ commu_meta.csv 복사 완료")

if not (DST / 'commu_midi').exists():
    print("MIDI 복사 중... (수 분 소요)")
    shutil.copytree(SRC / 'commu_midi', DST / 'commu_midi')
    print("✅ MIDI 복사 완료")
else:
    print("이미 복사되어 있습니다.")

## 6. Weights & Biases 연동

학습 로그와 loss curve를 기록합니다. 이 스크린샷들이 나중에
포트폴리오의 "실험을 체계적으로 관리했다"는 근거 자료가 됩니다.

[wandb.ai](https://wandb.ai)에서 가입 후 API 키를 발급받으세요.

In [ ]:
import wandb
wandb.login()   # API 키 입력 프롬프트

# 연동 테스트
run = wandb.init(project="music-ai-portfolio", name="phase0-setup-check",
                 config={"phase": 0, "note": "환경 세팅 검증"})
wandb.log({"setup_ok": 1})
run.finish()
print("✅ W&B 연동 확인 — 대시보드에서 phase0-setup-check 런을 확인하세요.")

## 7. 데이터 sanity check

여기서 확인할 것:
- 메타데이터 CSV의 컬럼이 `configs/tokenizer.yaml`의 스키마와 일치하는가
- MIDI 파일이 정상적으로 파싱되는가
- 논문에 적힌 분포(genre 2종, track_role 6종 등)와 실제 데이터가 맞는가

In [ ]:
import pandas as pd

META = '/content/drive/MyDrive/music-ai-portfolio/data/raw/commu_meta.csv'
df = pd.read_csv(META)

print("샘플 수:", len(df))
print("\n컬럼:")
for c in df.columns:
    print("  -", c)
df.head(3)

In [ ]:
# 메타데이터 분포 확인 — 논문 Figure 9와 대조
for col in ['genre', 'track_role', 'time_signature', 'rhythm', 'num_measures']:
    if col in df.columns:
        print(f"\n── {col} " + "─" * (30 - len(col)))
        print(df[col].value_counts())
    else:
        print(f"\n⚠ '{col}' 컬럼 없음 — 실제 컬럼명 확인 필요")

In [ ]:
# MIDI 파싱 테스트
import glob
import miditoolkit

files = sorted(glob.glob('/content/drive/MyDrive/music-ai-portfolio/data/raw/commu_midi/train/raw/*.mid'))
print("MIDI 파일 수:", len(files))

mid = miditoolkit.MidiFile(files[0])
print("\n파일:", files[0].split('/')[-1])
print("ticks_per_beat:", mid.ticks_per_beat)
print("악기 트랙 수  :", len(mid.instruments))
print("템포 변화     :", [(t.time, t.tempo) for t in mid.tempo_changes][:3])
print("박자표        :", [(s.time, f'{s.numerator}/{s.denominator}') for s in mid.time_signature_changes][:3])

notes = mid.instruments[0].notes
print(f"\n노트 수: {len(notes)}")
print("앞 5개 노트 (start, end, pitch, velocity):")
for n in notes[:5]:
    print(f"  {n.start:>6} {n.end:>6}  pitch={n.pitch:>3}  vel={n.velocity:>3}")

In [ ]:
# 노트 수 분포 — max_seq_len 결정 근거
import numpy as np
from tqdm.auto import tqdm

counts = []
for f in tqdm(files[:1000], desc="스캔"):
    try:
        m = miditoolkit.MidiFile(f)
        counts.append(sum(len(inst.notes) for inst in m.instruments))
    except Exception:
        pass

counts = np.array(counts)
print(f"\n노트 수 통계 (n={len(counts)})")
for p in [50, 90, 95, 99, 100]:
    print(f"  p{p:<3}: {np.percentile(counts, p):.0f}")
print("\n💡 REMI 계열은 노트 1개당 약 4~5 토큰이 필요합니다.")
print(f"   p99 기준 예상 토큰 수 ≈ {np.percentile(counts, 99) * 5:.0f}")
print("   → configs/base.yaml의 max_seq_len(현재 1024)이 충분한지 판단하세요.")

## 8. 세팅 결과 기록

아래 출력을 `docs/experiment_log.md`와 Notion 페이지에 붙여넣으세요.
Phase 2에서 하이퍼파라미터를 조정할 때 이 정보가 기준이 됩니다.

In [ ]:
from datetime import datetime
import torch

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음"
prec = "bf16" if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "fp16"

report = f"""
## Phase 0 세팅 완료 — {datetime.now():%Y-%m-%d %H:%M}

| 항목 | 값 |
|---|---|
| GPU | {gpu} |
| VRAM | {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB |
| 권장 precision | {prec} |
| Python | {__import__('sys').version.split()[0]} |
| torch | {torch.__version__} |
| 학습 샘플 수 | {len(df)} |
| 데이터 경로 | /content/drive/MyDrive/music-ai-portfolio/data/raw |

### 다음 액션
- [ ] configs/base.yaml의 precision을 `{prec}`로 확인/수정
- [ ] max_seq_len이 충분한지 위 노트 수 통계로 판단
- [ ] commu_meta.csv의 실제 컬럼명을 configs/tokenizer.yaml에 반영
- [ ] Phase 1 착수: data/parser.py 구현
"""
print(report)

---

## ✅ Phase 0 완료 조건 (DoD)

- [ ] Colab에서 GPU가 할당되고 torch가 CUDA를 인식함
- [ ] Drive 마운트 및 디렉토리 생성 완료
- [ ] requirements.txt의 모든 패키지 import 성공
- [ ] ComMU 데이터셋이 Drive에 복사됨 (train + val)
- [ ] W&B 대시보드에 테스트 런이 기록됨
- [ ] MIDI 파일 1개 이상이 정상 파싱됨
- [ ] 메타데이터 분포가 논문 기술과 대체로 일치함

전부 체크되면 **Phase 1(토크나이저)**로 넘어갑니다.